In [ ]:
import torch
import torch.nn as nn
from torchvision import transforms, datasets

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

from classifier_weather.src.dataset import get_transforms, TestDataset, build_idx_to_target
from classifier_weather.src.train import train, val
from classifier_weather.src.model import create_model
from classifier_weather.src.predict import save_predict

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
TRAIN_DATA_DIR = '../data/train/train'
TEST_DATA_DIR = '../data/test/test'

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

In [ ]:
full_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

base_dataset = datasets.ImageFolder(TRAIN_DATA_DIR, transform=full_transform)

targets = np.array([label for _, label in base_dataset.samples])

print(base_dataset.class_to_idx)
print(f"Всего изображений: {len(base_dataset)}")

In [ ]:
data_transforms = get_transforms(IMAGENET_MEAN, IMAGENET_STD)

train_size = int(0.8 * len(base_dataset))
val_size = len(base_dataset) - train_size

train_dataset, val_dataset = torch.utils.data.random_split(
    base_dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

train_idx = train_dataset.indices
val_idx = val_dataset.indices

train_dataset = torch.utils.data.Subset(datasets.ImageFolder(TRAIN_DATA_DIR, transform=data_transforms['train']), train_idx)
val_dataset = torch.utils.data.Subset(datasets.ImageFolder(TRAIN_DATA_DIR, transform=data_transforms['val']), val_idx)

train_dataloader = torch.utils.data.DataLoader(train_dataset, shuffle=True, batch_size=32, num_workers=2)
val_dataloader = torch.utils.data.DataLoader(val_dataset, shuffle=False, batch_size=32, num_workers=2)

In [ ]:
model = create_model(type_model='resnet18', unfreeze_last_block=True)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=model.fc.parameters(), lr=3e-4)

In [ ]:
def plot_training_curves(train_losses, train_scores, val_scores=None, save_path=None):
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    axes[0].plot(train_losses, label='Train Loss')
    axes[0].set_title('Loss по эпохам')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].legend()

    axes[1].plot(train_scores, label='Train F1')
    if val_scores is not None:
        axes[1].plot(val_scores, label='Val F1')
    axes[1].set_title('F1 по эпохам')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('F1 score')
    axes[1].legend()

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Сохранено: {save_path}")
    plt.show()

In [ ]:
model, train_losses, train_scores, val_scores = train(
    model, train_dataloader, val_dataloader, optimizer, criterion, num_epoch=25
)

plot_training_curves(
    train_losses, train_scores, val_scores,
    save_path='../results/training_curves_resnet18.png'
)

In [ ]:
test_dataset = TestDataset(TEST_DATA_DIR, transform=data_transforms['val'])

test_dataloader = torch.utils.data.DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0)

In [ ]:
class_to_target = {
    'rain': 0,
    'fog': 1,
    'snow': 2
}

model_idx_to_target = {
    base_dataset.class_to_idx[k]: v for k, v in class_to_target.items()
}

model.eval()

filenames = []
predictions = []

with torch.no_grad():
    for X_batch, names in tqdm(test_dataloader):
        X_batch = X_batch.to(device)
        outputs = model(X_batch)
        preds = outputs.argmax(dim=1)

        filenames.extend(names)
        predictions.extend([model_idx_to_target[p.item()] for p in preds])

save_predict(base_dataset, predictions, filenames, output_path='../results/resnet18_pred.csv')